<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day6/EnsembleLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/PML Day 6/devnagari_db_csv.zip')

In [3]:
df.shape

(92000, 1025)

In [4]:
df.columns

Index(['pixel_0000', 'pixel_0001', 'pixel_0002', 'pixel_0003', 'pixel_0004',
       'pixel_0005', 'pixel_0006', 'pixel_0007', 'pixel_0008', 'pixel_0009',
       ...
       'pixel_1015', 'pixel_1016', 'pixel_1017', 'pixel_1018', 'pixel_1019',
       'pixel_1020', 'pixel_1021', 'pixel_1022', 'pixel_1023', 'character'],
      dtype='object', length=1025)

In [5]:
df['character'].unique()

array(['character_01_ka', 'character_02_kha', 'character_03_ga',
       'character_04_gha', 'character_05_kna', 'character_06_cha',
       'character_07_chha', 'character_08_ja', 'character_09_jha',
       'character_10_yna', 'character_11_taamatar', 'character_12_thaa',
       'character_13_daa', 'character_14_dhaa', 'character_15_adna',
       'character_16_tabala', 'character_17_tha', 'character_18_da',
       'character_19_dha', 'character_20_na', 'character_21_pa',
       'character_22_pha', 'character_23_ba', 'character_24_bha',
       'character_25_ma', 'character_26_yaw', 'character_27_ra',
       'character_28_la', 'character_29_waw', 'character_30_motosaw',
       'character_31_petchiryakha', 'character_32_patalosaw',
       'character_33_ha', 'character_34_chhya', 'character_35_tra',
       'character_36_gya', 'digit_0', 'digit_1', 'digit_2', 'digit_3',
       'digit_4', 'digit_5', 'digit_6', 'digit_7', 'digit_8', 'digit_9'],
      dtype=object)

# Taking these characters for classification as they look similar

## 24 --> 25
## 17 --> 26
## 23 --> 29

In [10]:
df1 = df.loc[df['character'].isin(['character_17_tha','character_23_ba', 'character_24_bha',
       'character_25_ma', 'character_26_yaw','character_29_waw']), :]
df1.shape

(12000, 1025)

In [11]:
df1['character'].unique()

array(['character_17_tha', 'character_23_ba', 'character_24_bha',
       'character_25_ma', 'character_26_yaw', 'character_29_waw'],
      dtype=object)

In [17]:
X = df1.drop(columns = ['character'])
Y = df1['character']
X.shape, Y.shape

((12000, 1024), (12000,))

In [27]:
label_map = {'character_17_tha':0,
             'character_23_ba':1,
             'character_24_bha':2,
              'character_25_ma':3,
             'character_26_yaw':4,
             'character_29_waw':5}
Y.replace(label_map, inplace=True)
Y.unique()

/tmp/ipykernel_2381/3635866459.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Y.replace(label_map, inplace=True)
/tmp/ipykernel_2381/3635866459.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Y.replace(label_map, inplace=True)


array([0, 1, 2, 3, 4, 5])

# Classification task : Stratify the data
 - even if the data is balanced
 - data on which you have to stratify (i.e, Y)

# Random Sampling : Regression

In [28]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.3,
                                                    random_state=7,stratify=Y)
X_train.shape, X_test.shape, Y_train.shape, Y_test.shape

((8400, 1024), (3600, 1024), (8400,), (3600,))

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
lr = LogisticRegression(random_state=7)
lr.fit(X_train, Y_train)
Y_pred = lr.predict(X_test)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [20]:
print(classification_report(Y_test, Y_pred))

                  precision    recall  f1-score   support

character_17_tha       0.80      0.79      0.79       600
 character_23_ba       0.80      0.79      0.79       600
character_24_bha       0.82      0.86      0.84       600
 character_25_ma       0.80      0.79      0.80       600
character_26_yaw       0.71      0.70      0.70       600
character_29_waw       0.79      0.78      0.78       600

        accuracy                           0.79      3600
       macro avg       0.79      0.79      0.79      3600
    weighted avg       0.79      0.79      0.79      3600



In [21]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200,
                            oob_score=True,
                            max_samples=0.8,
                            max_features=1000,
                            random_state=7)
rf.fit(X_train, Y_train)
Y_pred = rf.predict(X_test)


In [22]:
print(classification_report(Y_test, Y_pred))

                  precision    recall  f1-score   support

character_17_tha       0.90      0.88      0.89       600
 character_23_ba       0.93      0.88      0.90       600
character_24_bha       0.93      0.88      0.90       600
 character_25_ma       0.89      0.90      0.90       600
character_26_yaw       0.84      0.92      0.88       600
character_29_waw       0.87      0.90      0.89       600

        accuracy                           0.89      3600
       macro avg       0.89      0.89      0.89      3600
    weighted avg       0.89      0.89      0.89      3600



# XGBoost

In [23]:
from xgboost import XGBClassifier

In [29]:
xgb = XGBClassifier(n_estimators=50,learning_rate=0.5,
                    max_depth=5,subsample=0.8,
                    random_state=7,)

xgb.fit(X_train, Y_train)
#Y_pred = xgb.predict(X_test)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.5, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=50,
              n_jobs=None, num_parallel_tree=None, ...)

In [31]:
Y_pred = xgb.predict(X_test)
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.94      0.93      0.94       600
           1       0.94      0.92      0.93       600
           2       0.95      0.93      0.94       600
           3       0.94      0.93      0.93       600
           4       0.91      0.95      0.93       600
           5       0.92      0.94      0.93       600

    accuracy                           0.93      3600
   macro avg       0.93      0.93      0.93      3600
weighted avg       0.93      0.93      0.93      3600



# Cat Boost

In [32]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [33]:
from catboost import CatBoostClassifier

In [34]:
cbc = CatBoostClassifier(iterations=100,learning_rate=0.2,
                    depth=5,cat_features=None,
                    random_state=7,loss_function='MultiClass')

cbc.fit(X_train, Y_train)
Y_pred = cbc.predict(X_test)

0:	learn: 1.5528045	total: 3.45s	remaining: 5m 41s
1:	learn: 1.4154003	total: 4.42s	remaining: 3m 36s
2:	learn: 1.3131167	total: 5.34s	remaining: 2m 52s
3:	learn: 1.2294930	total: 6.23s	remaining: 2m 29s
4:	learn: 1.1631525	total: 6.97s	remaining: 2m 12s
5:	learn: 1.1087566	total: 7.85s	remaining: 2m 3s
6:	learn: 1.0560095	total: 8.84s	remaining: 1m 57s
7:	learn: 1.0106893	total: 9.79s	remaining: 1m 52s
8:	learn: 0.9688586	total: 10.8s	remaining: 1m 49s
9:	learn: 0.9387975	total: 11.4s	remaining: 1m 43s
10:	learn: 0.9113573	total: 12.2s	remaining: 1m 38s
11:	learn: 0.8866678	total: 13.3s	remaining: 1m 37s
12:	learn: 0.8606281	total: 14.3s	remaining: 1m 35s
13:	learn: 0.8360881	total: 15.2s	remaining: 1m 33s
14:	learn: 0.8129978	total: 16.2s	remaining: 1m 32s
15:	learn: 0.7946848	total: 17.1s	remaining: 1m 29s
16:	learn: 0.7739366	total: 17.7s	remaining: 1m 26s
17:	learn: 0.7553594	total: 18.3s	remaining: 1m 23s
18:	learn: 0.7378757	total: 18.8s	remaining: 1m 20s
19:	learn: 0.7150768	to

In [35]:
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91       600
           1       0.92      0.87      0.89       600
           2       0.95      0.92      0.93       600
           3       0.90      0.92      0.91       600
           4       0.87      0.90      0.89       600
           5       0.89      0.91      0.90       600

    accuracy                           0.90      3600
   macro avg       0.91      0.90      0.90      3600
weighted avg       0.91      0.90      0.90      3600



## LightGBM Classifier

In [36]:
from lightgbm import LGBMClassifier

In [37]:
lgbm = LGBMClassifier(max_depth=5,learning_rate=0.2,
                      n_estimators=100,
                      subsample=0.8,
                      colsample_bytree=0.8,
                      random_state=7)
#
lgbm.fit(X_train, Y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.284049 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 159611
[LightGBM] [Info] Number of data points in the train set: 8400, number of used features: 784
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.2, max_depth=5,
               random_state=7, subsample=0.8)

In [38]:
Y_pred = lgbm.predict(X_test)
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95       600
           1       0.95      0.93      0.94       600
           2       0.97      0.94      0.96       600
           3       0.95      0.94      0.94       600
           4       0.93      0.96      0.94       600
           5       0.93      0.95      0.94       600

    accuracy                           0.95      3600
   macro avg       0.95      0.95      0.95      3600
weighted avg       0.95      0.95      0.95      3600



## Apply Voting Classifier & Bagging Classifier to change the model , pass manually

## Learning rate impact on performance

In [41]:
lgbm = LGBMClassifier(max_depth=5,learning_rate=0.2,
                      n_estimators=50,
                      subsample=0.8,
                      colsample_bytree=0.8,
                      random_state=7)
#
lgbm.fit(X_train, Y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.365675 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 159611
[LightGBM] [Info] Number of data points in the train set: 8400, number of used features: 784
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.2, max_depth=5,
               n_estimators=50, random_state=7, subsample=0.8)

In [42]:
Y_pred = lgbm.predict(X_test)
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.94      0.94      0.94       600
           1       0.94      0.90      0.92       600
           2       0.95      0.93      0.94       600
           3       0.93      0.92      0.92       600
           4       0.91      0.95      0.93       600
           5       0.91      0.94      0.92       600

    accuracy                           0.93      3600
   macro avg       0.93      0.93      0.93      3600
weighted avg       0.93      0.93      0.93      3600



In [43]:
lgbm = LGBMClassifier(max_depth=5,learning_rate=0.5,
                      n_estimators=50,
                      subsample=0.8,
                      colsample_bytree=0.8,
                      random_state=7)
#
lgbm.fit(X_train, Y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.382207 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 159611
[LightGBM] [Info] Number of data points in the train set: 8400, number of used features: 784
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.5, max_depth=5,
               n_estimators=50, random_state=7, subsample=0.8)

In [44]:
Y_pred = lgbm.predict(X_test)
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.93      0.94      0.94       600
           1       0.95      0.91      0.93       600
           2       0.96      0.93      0.95       600
           3       0.93      0.94      0.94       600
           4       0.91      0.94      0.92       600
           5       0.92      0.94      0.93       600

    accuracy                           0.93      3600
   macro avg       0.93      0.93      0.93      3600
weighted avg       0.93      0.93      0.93      3600



In [49]:
lgbm = LGBMClassifier(max_depth=5,learning_rate=0.5,
                      n_estimators=100,
                      subsample=0.8,
                      colsample_bytree=0.8,
                      random_state=7)
#
lgbm.fit(X_train, Y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.288465 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 159611
[LightGBM] [Info] Number of data points in the train set: 8400, number of used features: 784
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Info] Start training from score -1.791759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.5, max_depth=5,
               random_state=7, subsample=0.8)

In [50]:
Y_pred = lgbm.predict(X_test)
print(classification_report(Y_test, Y_pred))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95       600
           1       0.95      0.92      0.94       600
           2       0.95      0.94      0.95       600
           3       0.94      0.94      0.94       600
           4       0.92      0.95      0.94       600
           5       0.93      0.95      0.94       600

    accuracy                           0.94      3600
   macro avg       0.94      0.94      0.94      3600
weighted avg       0.94      0.94      0.94      3600

